# 02 · Очистка данных

Ноутбук очищает данные перед объединением и анализом.

- **Часть 1:** погода и модельный PM2.5 (CAMS) из Open-Meteo.
- **Часть 2:** PM2.5 станции (добавляется в задаче 2.2).

Порядок шагов — по Лекции 4: инспекция → имена → типы → дубликаты → пропуски → невозможные значения → выбросы → итоговые проверки.

Сырые файлы из `data/raw/` только читаем. Результат записываем в `data/processed/`. Запускать сверху вниз.

# Часть 1. Погода и CAMS

## Шаг 1. Импорты и пути

Подключаем pandas и наши модули `src/clean.py` и `src/collect.py`, задаём пути к файлам и списки колонок.

In [1]:
import sys

import pandas as pd

sys.path.append("..")
from src import clean, collect

WEATHER_RAW_PATH = "../data/raw/weather.csv"
CAMS_RAW_PATH = "../data/raw/cams_aq.csv"
WEATHER_CLEAN_PATH = "../data/processed/weather_clean.csv"
CAMS_CLEAN_PATH = "../data/processed/cams_clean.csv"

WEATHER_COLUMNS = list(collect.WEATHER_COLUMNS.values())
CAMS_COLUMNS = list(collect.CAMS_COLUMNS.values())

## Шаг 2. Инспекция

По Лекции 4 сначала смотрим на данные, потом исправляем. Подробный осмотр сделан в `01_collect` (шаги 12–15), здесь — короткая сводка: размер, типы, пропуски.

In [2]:
weather_raw = pd.read_csv(WEATHER_RAW_PATH)
cams_raw = pd.read_csv(CAMS_RAW_PATH)

for name, table in [("weather", weather_raw), ("cams", cams_raw)]:
    print(name, table.shape)
    print(table.dtypes)
    print(table.isna().sum())
    print()

weather_raw.describe()

weather (49104, 7)
time_utc          str
temp_c        float64
humidity        int64
precip        float64
pressure      float64
wind_speed    float64
wind_dir        int64
dtype: object
time_utc      0
temp_c        0
humidity      0
precip        0
pressure      0
wind_speed    0
wind_dir      0
dtype: int64

cams (28776, 2)
time_utc         str
cams_pm25    float64
dtype: object
time_utc     0
cams_pm25    0
dtype: int64



,temp_c,humidity,precip,pressure,wind_speed,wind_dir
count,49104.000000,49104.000000,49104.000000,49104.000000,49104.000000,49104.00000
mean,11.873153,55.073884,0.059883,1018.549723,1.563412,181.14897
std,11.913288,21.377138,0.323194,9.978545,0.910536,106.37092
min,-28.400000,7.000000,0.000000,994.600000,0.000000,1.00000
25%,2.400000,38.000000,0.000000,1010.800000,0.920000,101.00000
50%,13.000000,53.000000,0.000000,1017.500000,1.430000,169.00000
75%,21.300000,72.000000,0.000000,1025.400000,2.020000,270.00000
max,39.100000,100.000000,10.600000,1058.900000,10.740000,360.00000


## Шаг 3. Имена колонок

Имена проекта в snake_case заданы при выгрузке, значения не менялись. Модельный PM2.5 называется `cams_pm25`. Проверяем, что имена и порядок ожидаемые.

In [3]:
clean.check_columns(weather_raw, WEATHER_COLUMNS)
clean.check_columns(cams_raw, CAMS_COLUMNS)
print(list(weather_raw.columns))
print(list(cams_raw.columns))

['time_utc', 'temp_c', 'humidity', 'precip', 'pressure', 'wind_speed', 'wind_dir']
['time_utc', 'cams_pm25']


## Шаг 4. Типы: время в UTC

Сырые метки времени — строки без `Z`, поэтому `utc=True` и `format=` задаём явно. `errors="coerce"` превращает битые метки в NaT, а функция сразу считает их и падает с понятным сообщением. Числовые колонки уже числа.

In [4]:
weather = weather_raw.copy()
weather["time_utc"] = clean.parse_utc_times(weather_raw["time_utc"], collect.METEO_TIME_FORMAT)
cams = cams_raw.copy()
cams["time_utc"] = clean.parse_utc_times(cams_raw["time_utc"], collect.METEO_TIME_FORMAT)

for name, table in [("weather", weather), ("cams", cams)]:
    print(name, table["time_utc"].dtype, "NaT:", table["time_utc"].isna().sum())
    print("первая:", table["time_utc"].iloc[0], "последняя:", table["time_utc"].iloc[-1])

weather datetime64[us, UTC] NaT: 0
первая: 2020-04-09 00:00:00+00:00 последняя: 2025-11-14 23:00:00+00:00
cams datetime64[us, UTC] NaT: 0
первая: 2022-08-04 00:00:00+00:00 последняя: 2025-11-14 23:00:00+00:00


## Шаг 5. Местное время Алматы

Анализ ведём по `Asia/Almaty`. 1 марта 2024 пояс сменился с UTC+6 на UTC+5, поэтому смотрим стык: какие местные метки получаются из UTC-часов вокруг перехода.

In [5]:
junction = weather[
    (weather["time_utc"] >= pd.Timestamp("2024-02-29 16:00", tz="UTC"))
    & (weather["time_utc"] <= pd.Timestamp("2024-02-29 19:00", tz="UTC"))
]
pd.DataFrame({
    "time_utc": junction["time_utc"],
    "time_almaty": clean.to_almaty_time(junction["time_utc"]),
})

,time_utc,time_almaty
34120,2024-02-29 16:00:00+00:00,2024-02-29 22:00:00+06:00
34121,2024-02-29 17:00:00+00:00,2024-02-29 23:00:00+06:00
34122,2024-02-29 18:00:00+00:00,2024-02-29 23:00:00+05:00
34123,2024-02-29 19:00:00+00:00,2024-03-01 00:00:00+05:00


In [6]:
for name, table in [("weather", weather), ("cams", cams)]:
    local_naive = clean.to_almaty_time(table["time_utc"]).dt.tz_localize(None)
    print(name, "повторов местного времени без смещения:", local_naive.duplicated().sum())

weather повторов местного времени без смещения: 1
cams повторов местного времени без смещения: 1


Местная метка 23:00 29.02.2024 встречается дважды: один раз со смещением +06:00, второй раз +05:00. Поэтому ключом для объединения оставляем `time_utc` — он не прыгает. В файл местное время не пишем, а `clean.to_almaty_time` применяем при построении признаков (задача 3.1).

## Шаг 6. Дубликаты

Опасны дубликаты ключа: повторяющийся час сломает объединение. Считаем повторы `time_utc`.

In [7]:
print("weather дублей time_utc:", weather["time_utc"].duplicated().sum())
print("cams дублей time_utc:", cams["time_utc"].duplicated().sum())

weather дублей time_utc: 0
cams дублей time_utc: 0


## Шаг 7. Пропуски

Проверяем, что сетка часов полная (между первой и последней меткой нет «дыр») и что пустых значений нет. Если пропусков 0, интерполяция не нужна. Начало CAMS (2022-08-04) — это период источника, а не пропуск.

In [8]:
for name, table in [("weather", weather), ("cams", cams)]:
    first = table["time_utc"].min()
    last = table["time_utc"].max()
    expected_hours = int((last - first) / pd.Timedelta(hours=1)) + 1
    print(name, "строк:", len(table), "ожидаемо часов:", expected_hours)
    print(table.isna().sum())

weather строк: 49104 ожидаемо часов: 49104
time_utc      0
temp_c        0
humidity      0
precip        0
pressure      0
wind_speed    0
wind_dir      0
dtype: int64
cams строк: 28776 ожидаемо часов: 28776
time_utc     0
cams_pm25    0
dtype: int64


## Шаг 8. Невозможные значения

Границы `clean.VALID_RANGES` — физически возможные значения. Всё вне них — ошибка данных, а не выброс. Считаем нарушения по каждой колонке и требуем 0.

In [9]:
weather_bad = clean.count_out_of_range(weather, clean.VALID_RANGES)
cams_bad = clean.count_out_of_range(cams, clean.VALID_RANGES)
print("weather:", weather_bad)
print("cams:", cams_bad)
total_bad = sum(weather_bad.values()) + sum(cams_bad.values())
assert total_bad == 0, f"значения вне допустимых диапазонов: {total_bad}"

weather: {'temp_c': 0, 'humidity': 0, 'precip': 0, 'pressure': 0, 'wind_speed': 0, 'wind_dir': 0}
cams: {'cams_pm25': 0}


## Шаг 9. Выбросы по IQR

По Лекции 4 выбросы по IQR — «прожектор, а не приговор»: мы только считаем и смотрим, ничего не удаляем (пики смога и сильный ветер — реальные явления). Для `precip` IQR не считаем: больше половины часов без осадков, Q1 = Q3 = 0, и любой дождь стал бы «выбросом» (покажем квартили). Для `wind_dir` не считаем: это угол по кругу, 1° и 360° почти одно направление.

In [10]:
IQR_COLUMNS = ["temp_c", "humidity", "pressure", "wind_speed"]
for column in IQR_COLUMNS:
    mask = clean.iqr_outlier_mask(weather[column])
    print(column, "выбросов:", mask.sum(), "доля, %:", round(mask.mean() * 100, 3))

cams_mask = clean.iqr_outlier_mask(cams["cams_pm25"])
print("cams_pm25", "выбросов:", cams_mask.sum(), "доля, %:", round(cams_mask.mean() * 100, 3))

print("precip Q1:", weather["precip"].quantile(0.25), "Q3:", weather["precip"].quantile(0.75))

temp_c выбросов: 13 доля, %: 0.026
humidity выбросов: 0 доля, %: 0.0
pressure выбросов: 213 доля, %: 0.434
wind_speed выбросов: 1415 доля, %: 2.882
cams_pm25 выбросов: 1520 доля, %: 5.282
precip Q1: 0.0 Q3: 0.0


Посмотрим 5 самых больших значений модельного PM2.5 вместе с местным временем: это реальные пики зимнего смога, их оставляем.

In [11]:
top_cams = cams.nlargest(5, "cams_pm25").copy()
top_cams["time_almaty"] = clean.to_almaty_time(top_cams["time_utc"])
print(top_cams)

                      time_utc  cams_pm25               time_almaty
3731 2023-01-06 11:00:00+00:00      153.7 2023-01-06 17:00:00+06:00
3732 2023-01-06 12:00:00+00:00      147.1 2023-01-06 18:00:00+06:00
3730 2023-01-06 10:00:00+00:00      146.2 2023-01-06 16:00:00+06:00
3729 2023-01-06 09:00:00+00:00      132.7 2023-01-06 15:00:00+06:00
3733 2023-01-06 13:00:00+00:00      129.9 2023-01-06 19:00:00+06:00


## Шаг 10. Итоговые проверки

Контракт для объединения: один час UTC — одна строка, время по возрастанию с шагом 1 час, без пропусков и невозможных значений. Любое нарушение остановит ноутбук.

In [12]:
clean.check_clean_hourly(weather, WEATHER_COLUMNS)
clean.check_clean_hourly(cams, CAMS_COLUMNS)
print("проверки пройдены")

проверки пройдены


## Шаг 11. Сохраняем очищенные таблицы

Сырые файлы не трогаем. Очищенные пишем в `data/processed/` с явным форматом времени (смещение `+0000` говорит, что это UTC). Эти файлы перезаписываются при каждом запуске — это производные данные. Читаем их обратно и проверяем ещё раз.

In [13]:
clean.save_clean_csv(weather, WEATHER_CLEAN_PATH)
clean.save_clean_csv(cams, CAMS_CLEAN_PATH)

weather_check = clean.read_clean_csv(WEATHER_CLEAN_PATH)
cams_check = clean.read_clean_csv(CAMS_CLEAN_PATH)
clean.check_clean_hourly(weather_check, WEATHER_COLUMNS)
clean.check_clean_hourly(cams_check, CAMS_COLUMNS)
print("weather_clean:", weather_check.shape)
print("cams_clean:", cams_check.shape)
print(weather_check.head(3))
print(cams_check.head(3))

weather_clean: (49104, 7)
cams_clean: (28776, 2)
                   time_utc  temp_c  humidity  precip  pressure  wind_speed  \
0 2020-04-09 00:00:00+00:00     2.4        94     0.5    1026.8        2.33   
1 2020-04-09 01:00:00+00:00     2.2        99     0.5    1027.5        0.72   
2 2020-04-09 02:00:00+00:00     2.5        94     0.3    1028.1        1.14   

   wind_dir  
0       245  
1       326  
2       232  
                   time_utc  cams_pm25
0 2022-08-04 00:00:00+00:00       15.1
1 2022-08-04 01:00:00+00:00       15.9
2 2022-08-04 02:00:00+00:00       15.4


## Итог части 1

- **Что нашли:** пропусков, дублей и значений вне допустимых диапазонов нет (см. вывод шагов 7, 8 и 6); выбросы по IQR посчитаны и оставлены (шаг 9); местное время 23:00 29.02.2024 встречается дважды (шаг 5).
- **Что изменили:** только тип времени — строки стали датой с поясом UTC. Значения не менялись.
- **Что записали:** `data/processed/weather_clean.csv` и `data/processed/cams_clean.csv`.
- **Для задачи 2.2:** функции `parse_utc_times`, `to_almaty_time`, `count_out_of_range`, `iqr_outlier_mask`, `save_clean_csv` и `read_clean_csv` готовы.
- **Для задачи 3.1:** ключ объединения — `time_utc`; местное время считаем через `clean.to_almaty_time`; день 29.02.2024 по местному времени длится 25 часов.